In [0]:
%pip install shap scikit-learn sqlalchemy s3fs
dbutils.library.restartPython()


In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import shap
import s3fs
import pickle

import joblib 
from datetime import datetime, timedelta
from sklearn.metrics import  roc_auc_score
from sklearn.linear_model import LogisticRegression

from sqlalchemy import create_engine
from sqlalchemy import text

In [0]:
SAFETY_KEYPHRASES = [
    # Airbag
    "airbags deployed",
    "seat belts fail",
    "seat belts lock",
    "seat belts malfunctioned",

    # Braking system failures / ADAS braking issues
    "brake failure",
    "braking failed",
    "failed respond brake",
    "hard brake",
    "unsafe brakes sudden",
    "phantom braking",
    "automatic emergency braking",
    "emergency braking",
    "emergency braking activated",
    "emergency braking slammed",
    "false crash event",
    "falsely triggered crash",

    # Steering / control loss
    "power steering failed",
    "lost power steering",
    "steering failed",
    "steering stopped",
    "steering wheel locked",
    "steering wheel seized",
    "wheel lock",

    # Engine / powertrain critical failures
    "engine stalls",
    "engine shuts off",
    "engine cuts out",
    "engine seized",
    "engine overheating",
    "engine misfiring",
    "sudden loss power",
    "power loss driving",
    "sudden loss drive",

    # Transmission critical failures
    "transmission failed",
    "transmission died",
    "transmission engages delay",
    "delay transmission engaging",
    "transmission jerks decelerating",

    # Fire / electrical hazard
    "battery fire",
    "thermal runaway",
    "electrical fire",
    "electrical short",
    "burning smell electrical",

    # Fuel / fluid hazards
    "fuel leaking",
    "fuel pump failed",
    "coolant leaked engine",

    # ire / wheel failures
    "tire failure",
    "tire blew",
    "highway tire separated",

    # Structural failures (glass explosions)
    "sunroof shattered",
    "sunroof exploded",
    "sunroof spontaneously shattered",
    "sunroof glass shattered",
    "rear window exploded",
    "roof shattered",

    # Other critical mechanical failures
    "timing chain snapped",
]

SAFETY_KEYPHRASES = list(set(SAFETY_KEYPHRASES))

In [0]:
host = dbutils.secrets.get(scope="supabase", key="host")
database = dbutils.secrets.get(scope="supabase", key="db")
user = dbutils.secrets.get(scope="supabase", key="user")
password = dbutils.secrets.get(scope="supabase", key="password")
port = 6543

connection_string = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}?sslmode=require'
engine = create_engine(connection_string)

In [0]:
today = (datetime.now()).strftime("%Y-%m-%d")
complaint_path = f"s3://nhtsa-data-shak/bronze/complaint/complaints_{today}.csv"

try:
    dbutils.fs.ls(complaint_path)
except Exception:
    dbutils.notebook.exit(f"Notebook stopped: File not found at {complaint_path}")

df_spark = spark.read.csv(complaint_path, header=True, inferSchema=True)
complaints_s3 = df_spark.toPandas()

In [0]:
today = (datetime.now()).strftime("%Y-%m-%d")
recall_path = f"s3://nhtsa-data-shak/bronze/recall/recalls_{today}.csv"

df_spark = spark.read.csv(recall_path, header=True, inferSchema=True)
recall_s3 = df_spark.toPandas()

recall_s3[['MAKE', 'MODEL', 'YEAR']].drop_duplicates().to_sql(
        name='temp_upload', 
        con=engine, 
        if_exists='replace', 
        index=False
    )

delete_query = """
DELETE FROM recall
WHERE EXISTS (
    SELECT 1 
    FROM temp_upload 
    WHERE recall."MAKE" = temp_upload."MAKE"
        AND recall."MODEL" = temp_upload."MODEL"
        AND recall."YEAR" = temp_upload."YEAR"
);
"""

with engine.begin() as connection:
    connection.execute(text(delete_query))
    connection.execute(text("DROP TABLE temp_upload;"))
    recall_s3.to_sql('recall', engine, if_exists='append', index=False)

print(f'Uploaded {len(recall_s3)} rows to the database')

In [0]:
PASSENGER_MAKES = [
    "ACURA", "AUDI", "BMW",
    "BUICK", "CADILLAC", "CHEVROLET", "CHRYSLER", "DODGE", 
    "FORD", "GMC", "HONDA", "HUMMER", "HYUNDAI",
    "INFINITI", "JAGUAR", "JEEP", "KIA", "LAND ROVER",
    "LEXUS", "LINCOLN", "MAZDA",
    "MERCEDES-BENZ", "MERCEDES", "MINI",
    "MITSUBISHI", "NISSAN",
    "PORSCHE", "RAM",
    "SUBARU", "TESLA", "TOYOTA", "VOLKSWAGEN", "VOLVO"
]

complaints_s3 = complaints_s3.rename(columns = {
    "productMake": "MAKE",
    "productModel": "MODEL",
    "productYear": "YEAR",
    "dateComplaintFiled": "LDATE",
    "summary": "CDESCR"})

complaints_s3 = complaints_s3[complaints_s3['MAKE'].isin(PASSENGER_MAKES)]

complaints_s3 = complaints_s3[['MAKE', 'MODEL', 'YEAR', 'LDATE', 'CDESCR', 'ODINO']]

existing = pd.read_sql('''SELECT "ODINO" FROM complaints_raw;''', engine)
complaints_s3 = complaints_s3[~complaints_s3['ODINO'].isin(existing['ODINO'])]

complaints_s3.to_sql('complaints_raw', engine, if_exists='append', index=False)
print(f'Uploaded {len(complaints_s3)} rows to the database')

In [0]:
query = '''SELECT* FROM complaints_raw'''
with engine.connect() as connection:
    complaint_data_combined = pd.read_sql(query, connection)

In [0]:
query = '''SELECT* FROM recall'''
with engine.connect() as connection:
    recall_data_combined_test = pd.read_sql(query, connection)

In [0]:
# Ensure year is numeric column, filter out missing years
complaint_data_combined["YEAR"] = pd.to_numeric(complaint_data_combined["YEAR"], errors="coerce")
complaint_data_combined = complaint_data_combined.dropna(subset=["YEAR"])
complaint_data_combined["YEAR"] = complaint_data_combined["YEAR"].astype(int)
complaint_data_combined = complaint_data_combined.loc[complaint_data_combined['YEAR'] != 9999]

# Filter out maufacturer
complaint_data_combined.loc[complaint_data_combined["MAKE"] == 'MERCEDES', "MAKE"] = "MERCEDES-BENZ"
complaint_data_combined.loc[complaint_data_combined["MAKE"] == 'MERCEDES BENZ', "MAKE"] = "MERCEDES-BENZ"
complaint_data_combined = complaint_data_combined[complaint_data_combined["MAKE"].str.upper().isin(PASSENGER_MAKES)]

# Correct Date Format
numeric_dates = pd.to_numeric(complaint_data_combined["LDATE"], errors="coerce")
complaint_data_combined['LDATE'] = pd.to_datetime(complaint_data_combined['LDATE'].astype(str), format='%Y-%m-%d', errors='coerce')

# Determine if complaint is within first year of release (since some vehicles are released in the fall, complaint 4 months before start of new year are considered)
complaint_data_combined['RELEASE_DATE'] = pd.to_datetime(
    complaint_data_combined['YEAR'].astype(str) + '-01-01')

complaint_data_combined['within_12_months'] = (
    (complaint_data_combined['LDATE'] - complaint_data_combined['RELEASE_DATE']).dt.days.between(-180, 365)
)

# Drop duplicates for complaints from same person
complaint_data_combined = complaint_data_combined.drop_duplicates(subset = ["ODINO"])

In [0]:
# Complaints within first 12 months for each manufacturer
manufacturer = complaint_data_combined.groupby(["MAKE", "YEAR"]).agg(
    complaints_first_12m_make = ('within_12_months', 'sum')).reset_index() 

# Complaint data aggregated for each vehicle
complaint_data_combined = complaint_data_combined.groupby(["MAKE", "MODEL", "YEAR"]).agg(
    complaint_count = ("ODINO", "count"),
    first_complaint_date = ("LDATE", "min"),
    complaints_first_12m = ('within_12_months', 'sum'),
    median_mileage = ("MILES", "median"),
    description = ("CDESCR", lambda x: " ".join(x.dropna()))).reset_index()

complaint_data_combined['median_mileage'] = complaint_data_combined['median_mileage'].fillna(0)

# To ensure statistical accuracy, only vehicles with at least 10 complaints are included
complaint_data_combined = complaint_data_combined.loc[complaint_data_combined['complaint_count'] >= 7] 

complaint_data_combined = complaint_data_combined.merge(manufacturer, on = ['MAKE', 'YEAR'], how = 'left')

In [0]:
# For each vehicle, determine number of complaints in first 12 months compared to total complaints in first 12 months for manufacturer

alpha = 1.0
beta = 2.0

# Smooth the ratio so vehicles with low complaint volume have a ratio instead of hitting zero
complaint_data_combined['complaints_first_12m_ratio'] = (
    (complaint_data_combined['complaints_first_12m'] + alpha) / 
    (complaint_data_combined['complaints_first_12m_make'] + beta)
)

In [0]:
recall_data_combined_test['Recall'] = 1

recall_data_combined_test = recall_data_combined_test.rename(columns = {"MODEL YEAR": "YEAR"})
complaint_data_combined_test = complaint_data_combined[(complaint_data_combined["YEAR"] >= 2022) & (complaint_data_combined["YEAR"] <= 2026)]

complaint_recall_test = complaint_data_combined_test.merge(recall_data_combined_test[['MAKE', 'MODEL', 'YEAR', 'Recall', 'recall_count']], on = ['MAKE', 'MODEL', 'YEAR'], how = 'left') 
complaint_recall_test["Recall"] = pd.to_numeric(complaint_recall_test["Recall"], errors="coerce")
complaint_recall_test["Recall"] = complaint_recall_test["Recall"].fillna(0)
complaint_recall_test["Recall"] = complaint_recall_test["Recall"].astype(int)

In [0]:
# Each vehicle is scored by overlap of keywords

def keybert_safety_score(text):
    if not text:
        return 0
    text = str(text).lower()
    tokens = set(re.findall(r'\b\w+\b', text))
    
    score = 0
    for phrase in SAFETY_KEYPHRASES:
        phrase_tokens = set(phrase.lower().split())
        overlap = len(phrase_tokens & tokens) / len(phrase_tokens)
        if overlap >= 0.75:
            score += 1
    return score

In [0]:
complaint_recall_test["keybert_safety_score"] = complaint_recall_test["description"].apply(keybert_safety_score)
print(complaint_recall_test.groupby("Recall")[["keybert_safety_score"]].median())

In [0]:
import mlflow
lr_model_loaded = mlflow.sklearn.load_model("models:/workspace.default.lr_model_car_recall_project@production")
scaler_loaded = joblib.load('scaler.pkl')
lr_model_loaded.multi_class = "deprecated"
X_train_scaled = np.load('X_train_scaled.npy')

In [0]:
features = [
    'keybert_safety_score',
    "median_mileage",
    "complaints_first_12m_ratio"
]

X_final = complaint_recall_test[features]
y_final = complaint_recall_test["Recall"]

X_test_scaled = scaler_loaded.transform(X_final)

prob_recall = lr_model_loaded.predict_proba(X_test_scaled)[:, 1]
complaint_recall_test["Probability_Recall"] = prob_recall

In [0]:
masker = shap.maskers.Independent(X_train_scaled, max_samples=2434)
explainer = shap.LinearExplainer(
    lr_model_loaded, 
    masker,
    feature_names=features  
)
shap_values = explainer(X_test_scaled)

shap_df = pd.DataFrame(
    shap_values.values,     
    columns=features,
    index=X_final.index
)

for col in shap_df.columns:
    complaint_recall_test[f'shap_{col}'] = shap_df[col]

shap_keybert_safety_score = complaint_recall_test['shap_keybert_safety_score'].abs().median()
shap_median_mileage = complaint_recall_test['shap_median_mileage'].abs().median()
shap_complaints_first_12m_ratio = complaint_recall_test['shap_complaints_first_12m_ratio'].abs().median()


In [0]:
feature_display_names = {
    'keybert_safety_score': 'Keyword Score',
    'median_mileage': 'Median Mileage',
    'complaints_first_12m_ratio': 'Complaint Ratio (First 12 months)'
}

clean_feature_names = [feature_display_names[f] for f in features]

shap.summary_plot(
    shap_values.values, 
    X_test_scaled,
    feature_names=clean_feature_names,
    show=False
)
plt.xlim(-5, 5)

# Save the figure for MLflow logging
shap_fig = plt.gcf()
plt.savefig("shap_xgb_dot.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
result = complaint_recall_test[[
    "MAKE", "MODEL", "YEAR",
    "Recall", "Probability_Recall", "complaint_count", 
    "keybert_safety_score",
    "median_mileage",
    "complaints_first_12m", 'recall_count', 'first_complaint_date', 'shap_keybert_safety_score',
    'shap_median_mileage', 'shap_complaints_first_12m_ratio']] 


In [0]:
delete_query = """
DELETE FROM result
"""
with engine.begin() as connection:
    connection.execute(text(delete_query))

result.to_sql('result', engine, if_exists='append', index=False)

print(f'Uploaded {len(result)} rows to the database')

In [0]:
auc = roc_auc_score(y_final, prob_recall)

df = result.groupby('Recall').agg(
    median_keybert = ('keybert_safety_score', 'median')).reset_index()
no_recall_keybert = df.loc[df['Recall'] == 0, "median_keybert"]
recall_keybert= df.loc[df['Recall'] == 1, "median_keybert"]


In [0]:
df_eval = pd.DataFrame({
    'true_recall': y_final,
    'pred_prob': prob_recall  
})

df_eval['risk_bracket'] = pd.cut(df_eval['pred_prob'], 
                          bins=[0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                          labels=['Under 50', '50 – 60', '60 – 70', '70 – 80', '80 – 90', '90 – 100'])

probability_audit = df_eval.groupby('risk_bracket', observed=False).agg(
    total_cars=('true_recall', 'count'),
    actual_recalls=('true_recall', 'sum'),
    true_recall_rate=('true_recall', 'mean')
).reset_index()

import matplotlib.pyplot as plt

percentages = probability_audit['true_recall_rate'] * 100
fig, ax = plt.subplots()
bars = ax.bar(probability_audit['risk_bracket'], percentages)
ax.bar_label(bars, padding=3, fmt='%.2f%%')
ax.set_xticklabels(probability_audit['risk_bracket'], rotation=45)
ax.set_ylabel('Actual Recall Rate (%)') 
ax.set_xlabel('Predicted Risk Score (0 - 100 Index)')
ax.set_title('Predicted Risk vs Actual Recall Rate')
ax.set_ylim(0, max(percentages) * 1.15)

ax.legend()
plt.tight_layout()
plt.show()

dist_plot = fig

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow import MlflowClient

run_name = f"daily_monitoring_{datetime.now().strftime('%Y%m%d_%H%M')}"
with mlflow.start_run(run_name = run_name) as run:
    # Log metrics
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("no_recall_keybert", no_recall_keybert)
    mlflow.log_metric("recall_keybert", recall_keybert)
   
    mlflow.log_metric("shap_keybert_safety_score", shap_keybert_safety_score)
    mlflow.log_metric("shap_median_mileage", shap_median_mileage)
    mlflow.log_metric("shap_complaints_first_12m_ratio", shap_complaints_first_12m_ratio)
    
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "features": ", ".join(features),
        "min_complaints": 10
    })
    
    mlflow.log_figure(fig, "calibration_plot.png")
    mlflow.log_figure(shap_fig, "shap_summary_plot.png")
    
    calibration_record = probability_audit.copy()
    calibration_record['run_id'] = run.info.run_id
    calibration_record['run_date'] = datetime.now()
    calibration_record['is_baseline'] = False  # Set to True when deploying new model version
    client = MlflowClient()
    current_version = client.get_model_version_by_alias("workspace.default.lr_model_car_recall_project", "production").version
    calibration_record['model_version'] = current_version
    
    with engine.connect() as connection:
        calibration_record.to_sql(
            'calibration_history',
            con=connection,
            if_exists='append',
            index=False
        )
        
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"Registered as: workspace.default.vehicle_recall_risk")
    print(f"For Streamlit dashboard, use run_id: {run.info.run_id}")

In [0]:
'''signature = infer_signature(X_train_scaled, lr_model_loaded.predict(X_train_scaled))
model_info = mlflow.sklearn.log_model(
    sk_model=lr_model_loaded,
    artifact_path="model",
    input_example=X_test_scaled[:5],  # First 5 rows as example
    signature=signature,
    registered_model_name="workspace.default.vehicle_recall_risk"
)

# 3. NOW set the alias — model_info exists at this point
from mlflow import MlflowClient
client = MlflowClient()
client.set_registered_model_alias(
    "workspace.default.vehicle_recall_risk",
    "production",
    model_info.registered_model_version
)
print(f"Promoted version {model_info.registered_model_version} to production")'''